# sc-tumor-annotator — demo

Capability portrait, not a research result. This notebook walks through the
synthetic-data pipeline: generate a cohort, infer CNV, score, annotate, and
evaluate. Everything is deterministic.


In [ ]:
import numpy as np
from sctumor.synth import generate_multipatient, STROMAL_TYPES
from sctumor import cnv, evaluate

cohort = generate_multipatient(3, 700, seed=0)
print(cohort.n_cells, 'cells x', cohort.n_genes, 'genes')


In [ ]:
# CNV score: malignant cells should score higher than normal cells
ref = np.isin(cohort.obs['cell_type'].to_numpy(), STROMAL_TYPES)
mat, ch = cnv.infer_cnv(cohort.expr, cohort.genes, ref)
score = cnv.cnv_score(mat, ch)
mal = cohort.obs['is_malignant'].to_numpy()
print('malignant mean', round(float(score[mal].mean()), 3))
print('normal mean   ', round(float(score[~mal].mean()), 3))


In [ ]:
# 5-fold cross-validation, tree vs baseline
cv = evaluate.cross_validate(cohort, n_splits=5)
for k, v in cv['summary'].items():
    print(k, round(v['mean'], 3))
